# 03 — Rural Community Profiles: Jawa Barat (`32`) vs. Nusa Tenggara Barat (`52`)

**Project:** Justice for Rural Communities Observatory (JRC Observatory)  
**Data Tier:** `P1` Verified Public Aggregates (BPS 2024–2026 & Kemendesa Indeks Desa 2025).  
**Note:** Descriptive profile only; does not establish causality.

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path('..').resolve() if Path('../data').exists() else Path('.').resolve()
PUB = ROOT / 'data' / 'approved-public'

id_data = json.loads((PUB / 'kemendesa_p1_pilot_2025_indeks_desa.json').read_text(encoding='utf-8'))
ict_data = json.loads((PUB / 'bps_p1_pilot_2025_ict_digital.json').read_text(encoding='utf-8'))
pov_data = json.loads((PUB / 'bps_p1_pilot_2025_2026_poverty_hdi.json').read_text(encoding='utf-8'))
podes_data = json.loads((PUB / 'bps_p1_pilot_2024_2025_podes.json').read_text(encoding='utf-8'))

rows = []
for key, code, name in [('32_JAWA_BARAT', '32', 'Jawa Barat'), ('52_NUSA_TENGGARA_BARAT', '52', 'Nusa Tenggara Barat')]:
    prov_id = id_data['provinces'][key]
    upper_name = name.upper()
    inet = ict_data['tables']['Persentase Penduduk Berumur 5 Tahun ke Atas yang Mengakses Internet Selama Tiga Bulan Terakhir menurut Provinsi dan Tipe Daerah , 2025.csv']['pilot_rows'][upper_name]
    comp = ict_data['tables']['Persentase Penduduk yang Menggunakan Komputer menurut Provinsi dan Klasifikasi Daerah, 2025.csv']['pilot_rows'][upper_name]
    pov_row = pov_data['tables']['Jumlah dan Persentase Penduduk Miskin Menurut Provinsi, 2025.csv']['pilot_and_national_rows'][name]
    hdi_val = pov_data['tables']['[Metode Baru] Indeks Pembangunan Manusia (Umur Harapan Hidup Hasil Long Form SP2020), 2026.csv']['pilot_and_national_rows'][upper_name][0]
    rows.append({
        'province_code': code,
        'province': name,
        'indeks_desa_2025_mean': prov_id['mean_score'],
        'villages_submitted_2025': prov_id['villages_submitted'],
        'mandiri_pct': round(prov_id['status_counts']['mandiri'] / prov_id['villages_submitted'] * 100, 1),
        'poverty_sept_2025_pct': float(pov_row[5]),
        'hdi_2026': float(hdi_val),
        'rural_internet_2025_pct': float(inet[1]),
        'urban_internet_2025_pct': float(inet[0]),
        'rural_computer_2025_pct': float(comp[1]),
    })

profile_df = pd.DataFrame(rows)
print(profile_df.to_string(index=False))

In [ ]:
ntb_districts = pd.DataFrame(id_data['provinces']['52_NUSA_TENGGARA_BARAT']['districts_complete_8_of_8'])
ntb_districts['berkembang_pct'] = (ntb_districts['berkembang'] / ntb_districts['desa'] * 100).round(1)
print('\nNTB Indeks Desa 2025 by District (8/8 complete):')
print(ntb_districts.sort_values('berkembang_pct', ascending=False).to_string(index=False))